[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/snowflake/snowflake.ipynb)

# Snowflake Growth

This example computes a snowflake growing from a seed of ice in undercooled water into a six-armed dendrite. Water cooled below its melting point freezes onto any ice it touches, and freezing releases latent heat, which warms the water at the ice's edge and slows further freezing there. A bump on the edge reaches into colder water and sheds heat to more sides than the flat edge around it, so tips outrun sides and sprout branches of their own. A crystal's six-fold surface energy steers the fastest tips along six directions to form the arms of a snowflake. Entropy production drives the freezing because the latent heat released into colder water raises the water's entropy by more than the freezing lowers the ice's own entropy.

The algebra joins the projective plane `x+ y+ w0` with three degenerate directions `v0 i0 h0` for the quantities volume, ice and heat. A parcel of water is modelled as a sum of those units, one unit of volume plus its ice and its heat, and freezing is one fixed direction among the quantities, `freezing = ice + latent_heat * heat`. The energy plane `h - latent_heat * i` pairs with `freezing` to zero because freezing only moves energy from the latent store into heat, while the entropy slope pairs with `freezing` to give the entropy produced per unit of ice. Wedging minus a field's simplex gradient with its quantity direction forms a drive, and an isotropic material map turns drives into fluxes so `drive & spreading(drive)` measures squared edge steepness without taking a metric square. Multiplying the steepness by the crystal's axis forms a scaled rotor whose sixth power carries the six-fold anisotropy without computing an angle, with analytical derivative `6 * crystal_axis * turn ** 5`.

In vector-calculus notation, the interface normal angle $\theta = \operatorname{atan2}(\partial_y\phi, \partial_x\phi)$ enters $\epsilon(\theta) = \bar\epsilon\,(1 + \delta\cos 6\theta)$, and varying $\int \tfrac12\epsilon(\theta)^2|\nabla\phi|^2\,dA$ produces both the diffusive divergence and the skew Herring torque derivatives, $\tau\,\partial_t\phi = \nabla\cdot(\epsilon^2\nabla\phi) - \partial_x(\epsilon\,\epsilon'\,\partial_y\phi) + \partial_y(\epsilon\,\epsilon'\,\partial_x\phi) - 2W\phi(1-\phi)(1-2\phi) + L\,\tfrac{T_m - T}{T}\,p'(\phi)$, coupled to $\partial_t T = \nabla^2 T + L\,p'(\phi)\,\partial_t\phi$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples,jax] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import jax
import numpy as np

from numga import Algebra, stack
from numga.backend.jax import JaxContext
from numga.sparse import spdiag
from examples.util import shapes
from examples.util.animation import show_states
from examples.thermodynamics.snowflake import render, scenarios

jax.config.update("jax_enable_x64", True)

# The projective plane `x+ y+ w0` and three degenerate directions `v0 i0 h0` for volume, ice and heat.
space = Algebra("x+y+w0")
quantities = Algebra("v0i0h0")
ga = space * quantities
mv = JaxContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
# A cell integral, intrinsic like the join that measures a triangle's area.
Antiscalar = ga.gatype.antipseudoscalar()                                     # Antipseudoscalar-
Plane = ga.gatype.vector()                                                    # Vector+
Upright = Plane.derive.within(Algebra("x+y+"))                                # Vector+
Drive = ga.gatype.bivector().derive.touching(Algebra("x+y+"), Algebra("i0h0"))   # Bivector+
Flux = ga.gatype.antibivector().derive.touching(Algebra("x+y+"), Algebra("i0h0"))   # Antibivector-
origin = mv.w.dual()                                                          # [] Point-
directions = stack((mv.x.dual(), mv.y.dual()))                                # [2] Point-
volume = mv.v.dual()                                                          # [] Point-
ice = mv.i.dual()                                                             # [] Point-
heat = mv.h.dual()                                                            # [] Point-

## The dish of water

The water fills a round dish triangulated in staggered rings of nearly equilateral triangles, so the topology has no six-fold symmetry of its own. The water starts at temperature zero with a seed of ice at the centre, on a scale where ice melts at one. Two fields describe the state at the vertices: the parcels, and the phase, an order parameter equal to one in ice and zero in water whose smooth quintic step sets each parcel's ice fraction. Each triangle's corner simplex gradients are built from its sides by regressive joins, while triangle areas and vertex cell integrals keep intrinsic orientation.

In [ ]:
def ice_content(phase: Scalar) -> Scalar:
    """A parcel's ice fraction as a smooth quintic step between zero in water and one in ice, with vanishing slope at both."""
    return phase ** 3 * (10 - 15 * phase + 6 * phase ** 2)


# Unit-weight vertex places of the triangulated dish.
topology, coordinates = shapes.concentric_disk(np.array([scenarios.dish_radius]), scenarios.n_divisions)
vertices = (mv(Upright, coordinates) + mv.w).dual().field()                   # Point-[V]
phase = scenarios.seed_phase(vertices)                                        # Scalar+[V]
# Initial parcels of unit volume carrying the seed's ice fraction and zero heat.
parcels = volume + ice * ice_content(phase) + 0 * heat                        # Point-[V]

corners = topology.incidence[2, 0]                                            # [3] Scalar+[F <- V]
corner_points = corners * vertices                                            # [3] Point-[F]
facing = topology.off[2, 0] * vertices                                        # [2, 3] Point-[F]
sides = facing[0] & facing[1]                                                 # [3] Antibivector-[F]
unread = volume & ice & heat                                                  # [] Antitrivector-
# Joining all three corners with unit volume, ice and heat gives twice the oriented triangle area.
double_areas = corner_points[0] & corner_points[1] & corner_points[2] & unread   # Antiscalar-[F]
areas = double_areas / 2                                                      # Antiscalar-[F]
# Joining the facing side with volume, ice and heat over the corner's join gives each corner's spatial simplex gradient.
gradients = ((sides & unread) / (sides & unread & corner_points)).cast(Upright)   # [3] Upright+[F]
dual_areas = (corners.adjugate() * (areas / 3)).sum(axis=0)                   # Antiscalar-[V]
render.draw_seed(topology, vertices, parcels);

In the close-up, the pale seed of ice sits a fifth of a unit across at the centre of the dark cold water. The innermost ring is a hexagon, and each outward ring adds vertices so the triangulation quickly falls out of step with any six-fold pattern.

## Freezing and its entropy

Freezing shifts a parcel along one direction among the quantities, `freezing = ice + latent_heat * heat`. Ice is taken at water's density, so freezing leaves the volume unchanged (`v & freezing == 0`), and ice and water share one heat capacity and conductivity. The energy plane `h - latent_heat * i` reads a parcel's heat minus the latent heat released by its ice and pairs with `freezing` to zero (`energy & freezing == 0`). The entropy slope across the quantities pairs with `freezing` to the entropy produced per unit of ice, positive in undercooled water and zero at the melting point. Because water's latent heat warms it by about 80 kelvin, a latent heat of 1.8 undercoolings corresponds to an undercooling near 44 kelvin and places the melting point six undercoolings above absolute zero.

In [ ]:
# Latent heat per unit ice, melting temperature above the initial water temperature, and absolute zero.
latent_heat, melting_temperature, absolute_zero = 1.8, 1.0, -5.0
absolute_melting = melting_temperature - absolute_zero
freezing = ice + latent_heat * heat                                           # [] Point-
energy = mv.h - latent_heat * mv.i                                            # [] Plane+

## The edge's free energy

The phase carries the free energy of the ice's edge across the dish, combining a double well that has minima at water and ice with a gradient energy over the triangles that penalizes steep interfaces and sets their width. Divided by the melting temperature, that free energy reduces the total entropy. Pairing the phase's drive with its flux `drive & spreading(drive)` measures squared steepness without a metric square, while the sixth power of `steepness * crystal_axis` varies the interface width six-fold with normal direction.

In [ ]:
# Relaxation time, mean interface width, six-fold width anisotropy, steepness regularization, and double-well height.
relaxation_time, mean_width, anisotropy, regularization, well_height = 3e-4, 0.0082, 0.025, 1.0, 1 / 6
crystal_axis = mv.y                                                           # [] Upright+

In [ ]:
# Simplex gradient operator from vertex scalars to triangle planes, and isotropic material map for heat and ice.
gradient = (spdiag(gradients) * corners).sum(axis=0) * Scalar                 # (Upright+ <- Scalar+)[F <- V]
sweeps = stack((directions & heat, directions & ice))                         # [2, 2] Flux-
spreading = (sweeps * (sweeps & Drive)).sum(axis=-1).sum(axis=0)              # [] Flux- <- Drive+

The derivative of the edge's free energy with respect to the vertex phases holds the interface back. Differentiating the six-fold polynomial gives the fifth power `6 * crystal_axis * turn ** 5`, matching the Cahn-Hoffman vector of an anisotropic surface energy, and each vertex receives the pairing of its corner drive with the resulting flux. That turn `steepness * crystal_axis` is the one place where the plane's Euclidean metric enters, because crystal angles live in `x+ y+`; its plane-valued derivative is wedged with `i` and converted into a flux by the isotropic material map.

In [ ]:
corner_drives = -gradients ^ mv.i                                             # [3] Drive+[F]


def edge_slope(phase: Scalar, anisotropy: float) -> Antiscalar:
    """The derivative of the edge's free energy with respect to the vertex phases."""
    steepness = gradient(phase)                                               # Upright+[F]
    # Pairing the phase's drive with its flux gives the squared steepness, whose derivative is twice that flux.
    drive = -steepness ^ mv.i                                                 # Drive+[F]
    flux = spreading(drive)                                                   # Flux-[F]
    squared_steepness = drive & flux                                          # Scalar+[F]
    turn = steepness * crystal_axis                                           # Even+[F]
    third_power = turn ** 3                                                   # Even+[F]
    six_fold = third_power.scalar_product(third_power)                        # Scalar+[F]
    reach = squared_steepness + regularization ** 2                           # Scalar+[F]
    widths = mean_width * (1 + anisotropy * six_fold / reach ** 3)            # Scalar+[F]
    # Differentiating the six-fold term gives `6 * crystal_axis * turn ** 5`, mapped to a flux by `spreading`.
    slope = 6 * crystal_axis * turn ** 5                                      # Upright+[F]
    slope_flux = spreading(-slope ^ mv.i)                                     # Flux-[F]
    # Combine the width's angular variation with the isotropic flux to form the total flux pull on each triangle.
    turning = mean_width * anisotropy * (slope_flux / reach ** 3 - 6 * six_fold * flux / reach ** 4)   # Flux-[F]
    pull = widths ** 2 * flux + squared_steepness * widths * turning          # Flux-[F]
    # Pair each corner's simplex drive with the triangle's flux pull to gather the vertex derivatives.
    shares = (corners.adjugate() * (areas * (corner_drives & pull))).sum(axis=0)   # Antiscalar-[V]
    well_slope = 2 * well_height * phase * (1 - phase) * (1 - 2 * phase)      # Scalar+[V]
    return shares + dual_areas * well_slope

## Growing the snowflake

The phase relaxes up the total entropy at fixed energy over its relaxation time. Because the quintic step's slope vanishes at zero and one, the thermodynamic push vanishes in pure water, keeping the cold water metastable until the ice front reaches it. At each step every parcel updates its ice fraction along the freezing direction, releasing latent heat, while heat conduction transports warmth between neighbouring vertex cells across an insulated rim. This cell and the next take about fifteen seconds, most of it stepping the water.

In [ ]:
dt, stride = 2e-4, 20
n_snowflake_frames = 75
frame_duration_ms = 40

# Assembled heat conduction operator pulling `spreading` back through the corner simplex gradients.
heat_drives = -gradients ^ mv.h                                               # [3] Drive+[F]
local = areas * (mv.h & (gradients[:, None] ^ spreading(heat_drives[None, :])))   # [3, 3] Antiscalar-[F]
conduction = (corners[:, None].adjugate() * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)   # Antiscalar-[V <- V]

In [ ]:
def growth(parcels: Point, phase: Scalar, anisotropy: float, n_frames: int) -> Iterator[tuple[Point, Scalar]]:
    """Yield parcel and phase fields across frames for a given six-fold width anisotropy."""

    def step(state: tuple[Point, Scalar], _) -> tuple[tuple[Point, Scalar], None]:
        parcels, phase = state
        temperatures = mv.h & parcels                                         # Scalar+[V]
        # Heat raises entropy by `1 / T`, while ice carries `latent_heat / T_m` less entropy than liquid water.
        absolute_temperatures = temperatures - absolute_zero                  # Scalar+[V]
        entropy_slope = mv.h / absolute_temperatures - latent_heat / absolute_melting * mv.i   # Plane+[V]
        # Pairing `entropy_slope` with `freezing` gives the entropy produced per unit of ice.
        held_slope = 30 * phase ** 2 * (1 - phase) ** 2                       # Scalar+[V]
        push = absolute_melting * dual_areas * (entropy_slope & freezing) * held_slope   # Antiscalar-[V]
        next_phase = phase + dt * (push - edge_slope(phase, anisotropy)) / (relaxation_time * dual_areas)   # Scalar+[V]
        ice_gained = ice_content(next_phase) - (mv.i & parcels)               # Scalar+[V]
        conducted = -(conduction * temperatures) / dual_areas                 # Scalar+[V]
        return (parcels + ice_gained * freezing + dt * conducted * heat, next_phase), None

    @jax.jit
    def advance(state: tuple[Point, Scalar]) -> tuple[Point, Scalar]:
        return jax.lax.scan(step, state, None, length=stride)[0]

    state = (parcels, phase)
    for _ in range(n_frames):
        yield state
        state = advance(state)


snowflake = show_states(growth(parcels, phase, anisotropy, n_snowflake_frames), render.animate, topology, vertices,
                        name="snowflake", duration_ms=frame_duration_ms)

The seed first forms a hexagonal plate with corners on the six crystal axes and faces turned between them where the interface costs least. Those corners reach into colder water and grow into six primary arms whose sides sprout secondary branches along the same axes, surrounded by a lighter blue halo of latent heat diffusing into the dark cold water.

## Entropy over the growth

Total entropy sums the parcels' entropy across the cells and subtracts the edge's free energy divided by the melting temperature. Stepping along the freezing direction and conducting heat across an insulated rim conserve total energy by construction, while phase relaxation and heat conduction from warm to cold water steadily raise the total entropy.

In [ ]:
def edge_free_energy(phase: Scalar, anisotropy: float) -> Antiscalar:
    """The free energy of the ice's edge, summing the anisotropic gradient energy and the double well."""
    steepness = gradient(phase)                                               # Upright+[F]
    drive = -steepness ^ mv.i                                                 # Drive+[F]
    squared_steepness = drive & spreading(drive)                              # Scalar+[F]
    third_power = (steepness * crystal_axis) ** 3                             # Even+[F]
    six_fold = third_power.scalar_product(third_power)                        # Scalar+[F]
    widths = mean_width * (1 + anisotropy * six_fold / (squared_steepness + regularization ** 2) ** 3)   # Scalar+[F]
    well = well_height * phase ** 2 * (1 - phase) ** 2                        # Scalar+[V]
    return (areas * widths ** 2 * squared_steepness / 2).sites.sum() + (dual_areas * well).sites.sum()


def total_entropy(parcels: Point, phase: Scalar) -> Antiscalar:
    """The total entropy of the dish from the parcels' entropy and the edge's free energy."""
    parcel_entropies = ((mv.h & parcels) - absolute_zero).log() - latent_heat / absolute_melting * (mv.i & parcels)   # Scalar+[V]
    return (dual_areas * parcel_entropies).sites.sum() - edge_free_energy(phase, anisotropy) / absolute_melting


entropies = stack([total_entropy(*state) for state in snowflake])             # [n_snowflake_frames] Antiscalar-
render.draw_entropy(dt * stride * np.arange(n_snowflake_frames), entropies);

In the chart, the entropy gained since the start rises throughout the run and accelerates as the branching perimeter lengthens and releases latent heat faster.

## Growth without anisotropy

Running the same seed with zero anisotropy makes the interface width uniform in every direction until the tips again reach about nine tenths of the dish radius. This cell takes about twenty seconds, most of it stepping the water.

In [ ]:
n_isotropic_frames = 111
isotropic = show_states(growth(parcels, phase, 0.0, n_isotropic_frames), render.animate, topology, vertices,
                        name="snowflake_isotropic", duration_ms=frame_duration_ms)

Without crystal anisotropy the interface remains unstable, so bumps still outrun flat segments and split into fingers. With no preferred axes, small irregularities in the triangulation seed an irregular bush of fingers that takes roughly half again as long to cross the dish, confirming that the snowflake's six arms come from the crystal's six-fold surface energy.

In [ ]:
# checks
# Freezing moves energy from the latent store into heat, so the energy plane pairs with `freezing` to zero.
np.testing.assert_allclose((energy & freezing).to_array(), 0.0, atol=1e-15)
# With an insulated rim, total energy summed across the cells stays constant in both runs.
for states in (snowflake, isotropic):
    first, last = ((dual_areas * (energy & parcels)).sites.sum().to_array() for parcels, _ in (states[0], states[-1]))
    np.testing.assert_allclose(last, first, rtol=1e-10)
# Total entropy rises monotonically across every recorded frame.
assert np.all(np.diff(entropies.to_array()) > 0)
# Early in the run, the anisotropic crystal aligns its facets between the six crystal axes whereas the isotropic seed does not.
plate_frame = 2
alignments = []
for states in (snowflake, isotropic):
    steepness = gradient(states[plate_frame][1])                              # Upright+[F]
    squared_steepness = (-steepness ^ mv.i) & spreading(-steepness ^ mv.i)    # Scalar+[F]
    third_power = (steepness * crystal_axis) ** 3                             # Even+[F]
    six_fold = third_power.scalar_product(third_power)                        # Scalar+[F]
    alignments.append(((areas * six_fold / (squared_steepness + regularization ** 2) ** 2).sites.sum()
                       / (areas * squared_steepness).sites.sum()).to_array())
assert alignments[0] < -0.25
assert abs(alignments[1]) < 0.1